In [0]:
from pyspark.sql.functions import col, rand, floor, lit, concat, explode, array
import pyspark.sql.functions as F

# ==========================================
# 1. AQE (Adaptive Query Execution) - Podejście nowoczesne
# ==========================================
# UWAGA ARCHITEKTONICZNA: W architekturze Databricks Serverless i Spark Connect, 
# mechanizm AQE jest natywnie wymuszony i system blokuje ręczną zmianę tych parametrów.
# W klasycznym klastrze użylibyśmy poniższych komend do obsługi Data Skew:
# spark.conf.set("spark.sql.adaptive.enabled", "true")
# spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
# spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "8mb") 

print("✅ Mechanizm AQE jest natywnie zarządzany przez platformę Databricks Serverless!")

# ==========================================
# 2. Przygotowanie danych do demonstracji
# ==========================================
payroll_df = spark.read.table("dbw_showcase.default.payroll_silver")

# Tworzymy małą, sztuczną tabelę wymiarów (Dimension) - np. budżety biur
offices_data = [
    ("Krakow HQ", 1000000), ("Wroclaw", 500000), 
    ("Warsaw", 750000), ("Gdansk", 400000), ("Poznan", 300000)
]
offices_dim = spark.createDataFrame(offices_data, ["office_location", "budget"])

# ==========================================
# 3. SALTING - Klasyczna inżynieria rozpraszania kluczy
# ==========================================
print("⏳ Przygotowuję Salting dla potężnego Data Skew (Krakow HQ to ok. 80% danych)...")

SALT_BINS = 5 # Rozbijamy przeciążony klucz na 5 mniejszych części (kubełków)

# Krok A: "Solenie" dużej tabeli (Fact Table) - dodajemy losową liczbę (0 do 4) do nazwy biura
payroll_salted = payroll_df.withColumn(
    "salted_office", 
    concat(col("office_location"), lit("_"), floor(rand() * SALT_BINS))
)

# Krok B: Replikacja małej tabeli (Dimension Table) 
# Zamiast powolnych pętli w Pythonie (UDF), używamy natywnego `explode` na wygenerowanej tablicy!
salt_array = array(*[lit(i) for i in range(SALT_BINS)])
offices_salted = offices_dim.withColumn("salt", explode(salt_array)) \
                            .withColumn("salted_office", concat(col("office_location"), lit("_"), col("salt")))

# ==========================================
# Krok C: Optymalny Skew Join z ALIASAMI (Rozwiązanie błędu AMBIGUOUS_REFERENCE)
# ==========================================
# Nadajemy tabelom krótkie nazwy: "fact" (tabela główna) i "dim" (tabela słownikowa)
optimized_join_df = payroll_salted.alias("fact").join(
    offices_salted.alias("dim"),
    "salted_office",
    "inner"
)

# 4. Wyświetlamy wynik złączenia 
# Teraz wyraźnie mówimy Sparkowi: Pogrupuj po kolumnie 'office_location', ale tej z tabeli 'fact'!
display(
    optimized_join_df.groupBy("fact.office_location").agg(
        F.count("*").alias("transaction_count"),
        F.first("budget").alias("office_budget")
    ).orderBy(F.desc("transaction_count"))
)